# Google Earth Engine Data Acquisition

## Import Libraries

In [12]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt
#import ee
from shapely.geometry import mapping
import pandas as pd

## Initialize GEE

In [3]:
# Trigger the GEE authentication
ee.Authenticate()

# Initialize the cloud project
ee.Initialize(project='ee-stidjaco')

## Set Variables

In [13]:
# Set origin path for GM-SEUS and GM-SEUS GEE Asset Path
gmseusLocalPath = r"S:\Users\stidjaco\R_files\BigPanel\Data\Derived\GMSEUS\SHP\GMSEUS_Arrays_Final_2025_v2_1.shp"
gmseusGEEAssetPath = r"projects/ee-stidjaco/assets/BigPanel/GMSEUS_Arrays_Final_2025_v2_1" # Upload prior to running this script

# Set GDrive output path
gdriveOutputPath = r'H:\My Drive'

# Set Helper Functions

In [ ]:
# Function to convert a GeoPandas GeoDataFrame row to an Earth Engine Feature (back-up, ideally, object is already a GEE Asset)
def gdf_row_to_ee_feature(row, source_crs="EPSG:6350"):
    """
    Convert a GeoPandas GeoDataFrame row to an Earth Engine Feature.

    Args:
        row (GeoSeries): A row from a GeoPandas GeoDataFrame.
        source_crs (str): The coordinate reference system of the input row.

    Returns:
        ee.Feature: An Earth Engine Feature representing the input row.
    """
    gdf = gpd.GeoDataFrame(
        [row],
        geometry="geometry",
        crs=source_crs
    ).to_crs("EPSG:4326")

    row = gdf.iloc[0]

    return ee.Feature(
        ee.Geometry(mapping(row.geometry)),
        row.drop("geometry").to_dict()
    )

# Function to get land use / land cover classes for a solar array feature in the USA
def getGEE_lulc_usa(feature):
    """
    Assign pre-installation land use / land cover classes to a solar array feature.

    This function queries annual LULC datasets for up to 5 years prior to the
    installation year and assigns the dominant CDL-derived class within the
    buffered array footprint.

    Current outputs:
        cdl_yr1, cdl_yr2, cdl_yr3, cdl_yr4, cdl_yr5

    Notes:
        - CDL is used as the primary LULC product.
        - Pre-2008 CDL years are backfilled using NLCD cultivated classes.
        - CDL Grass/Pasture class 176 is split using Global Pasture Watch.
        - Missing values are assigned -9999.
    """

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Load land cover products
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    cdl = ee.ImageCollection("USDA/NASS/CDL")
    nlcd = ee.ImageCollection("projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER")
    gpw = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Set land cover product metadata
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    cdl_bnds = [2008, 2025]
    cdl_band = "cropland"
    cdl_res = 30
    cdl_earliest_start = "2008-01-01"
    cdl_earliest_end = "2008-12-31"

    nlcd_bnds = [1985, 2025]
    nlcd_band = "b1"
    nlcd_res = 30
    nlcd_earliest_start = "1985-01-01"
    nlcd_earliest_end = "1985-12-31"

    gpw_bnds = [2000, 2022]
    gpw_band = "dominant_class"
    gpw_res = 30
    gpw_earliest_start = "2000-01-01"
    gpw_earliest_end = "2000-12-31"

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Get a single annual LULC image with date and availability checks
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    def get_lulc_map(
        img_col,
        aoi,
        yod,
        lulc_bnds,
        lulc_band,
        earliest_start,
        earliest_end
    ):
        """
        Return a single-band LULC image for a requested year.

        The requested year is clamped to the available date range for the
        dataset. If no image is available, returns a constant -9999 image.
        """

        lulc_year = ee.Number(yod).clamp(lulc_bnds[0], lulc_bnds[1])
        year_diff = lulc_year.subtract(lulc_bnds[0])

        start_date = ee.Date(earliest_start).advance(year_diff, "year")
        end_date = ee.Date(earliest_end).advance(year_diff, "year")

        lulc_col = (
            img_col
            .select(lulc_band)
            .filterDate(start_date, end_date)
            .filterBounds(aoi)
        )

        lulc_map = ee.Image(
            ee.Algorithms.If(
                lulc_col.size().gt(0),
                lulc_col.mode(),
                ee.Image.constant(-9999)
            )
        ).rename("lulc").toInt16()

        return lulc_map

        # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Get dominant CDL class within the array footprint
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    def get_lulc_val(aoi, yod):
        """
        Get the modal CDL-derived LULC class within an AOI for one year.

        Corrections applied:
            1. Pre-2008 CDL gap handling using NLCD cultivated classes.
            2. Grass/Pasture class 176 splitting using Global Pasture Watch.
                a. Cultivated grassland = 1000
                b. Natural grassland = 1001

        Returns:
            Integer LULC class code.
        """

        # ~~~~~~~~ Get CDL image for requested year

        cdl_filtered = get_lulc_map(
            cdl,
            aoi,
            yod,
            cdl_bnds,
            cdl_band,
            cdl_earliest_start,
            cdl_earliest_end
        )

        # ~~~~~~~~ Backfill pre-2008 CDL using NLCD cultivated classes

        def pre_2008_cdl_fix():

            nlcd_filtered = get_lulc_map(
                nlcd,
                aoi,
                yod,
                nlcd_bnds,
                nlcd_band,
                nlcd_earliest_start,
                nlcd_earliest_end
            )

            is_cult_nlcd = (
                nlcd_filtered.eq(82)
                .Or(nlcd_filtered.eq(81))
            )

            return cdl_filtered.where(
                is_cult_nlcd.Not(),
                88
            )

        cdl_filtered = ee.Image(
            ee.Algorithms.If(
                ee.Number(yod).lt(cdl_bnds[0]),
                pre_2008_cdl_fix(),
                cdl_filtered
            )
        )

        # ~~~~~~~~ Split CDL Grass/Pasture using Global Pasture Watch

        gpw_filtered = get_lulc_map(
            gpw,
            aoi,
            yod,
            gpw_bnds,
            gpw_band,
            gpw_earliest_start,
            gpw_earliest_end
        )

        cdl_grass_pasture = cdl_filtered.eq(176)

        is_cult = gpw_filtered.eq(1)
        is_nat = gpw_filtered.eq(2)

        cdl_filtered = (
            cdl_filtered
            .where(cdl_grass_pasture, 1002)                 # Unknown grassland
            .where(cdl_grass_pasture.And(is_cult), 1000)    # Cultivated grassland
            .where(cdl_grass_pasture.And(is_nat), 1001)     # Natural grassland
        )

        # ~~~~~~~~ Extract modal land cover class within AOI

        lulc_w_null = ee.Dictionary(
            cdl_filtered.reduceRegion(
                reducer=ee.Reducer.mode(maxRaw=1e5),
                geometry=aoi,
                scale=cdl_res,
                bestEffort=True,
                maxPixels=1e13
            )
        )

        # ~~~~~~~~ Handle null values

        lulc = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(
                    lulc_w_null.get("lulc"),
                    None
                ),
                -9999,
                lulc_w_null.get("lulc")
            )
        ).toInt()

        return lulc

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Set AOI and installation year
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    aoi = feature.geometry().buffer(5, ee.ErrorMargin(1))

    null_inst_yr = 2022

    inst_yr_raw = feature.get("instYr")

    inst_yr_checked = ee.Algorithms.If(
        ee.Algorithms.IsEqual(inst_yr_raw, None),
        null_inst_yr,
        inst_yr_raw
    )

    inst_yr = ee.Number(
        ee.Algorithms.If(
            ee.Number(inst_yr_checked).eq(-9999),
            null_inst_yr,
            inst_yr_checked
        )
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Get CDL classes for multiple years prior to installation
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    lulc_props = {}

    for year_prior in range(1, 6):

        yod = inst_yr.subtract(year_prior)

        lulc_props[f"lulc_yr{year_prior}"] = get_lulc_val(aoi, yod)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Return feature with new LULC attributes
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    return feature.set(lulc_props)

# Function to get irrigation status for a solar array feature in the USA
def getGEE_irrigStatus_usa(feature):
    """
    Assign pre-installation irrigation status to a solar array feature.

    This function queries LANID irrigation maps for up to 5 years prior to
    installation and determines whether irrigation was present within the
    array footprint.

    Outputs:
        irrig_yr1
        irrig_yr2
        irrig_yr3
        irrig_yr4
        irrig_yr5

    Values:
        1 = Irrigated
        0 = Not irrigated / no irrigation detected

    Notes:
        - LANID available from 1997-2017.
        - Requested years are clamped to dataset bounds.
        - Irrigation is detected using Reducer.max().
    """

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Load irrigation dataset
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    lanid = (
        ee.Image("users/xyhuwmir4/LANID/LANID_v1_rse")
        .unmask(0)
        .select(
            [
                "irMap97", "irMap98", "irMap99",
                "irMap00", "irMap01", "irMap02",
                "irMap03", "irMap04", "irMap05",
                "irMap06", "irMap07", "irMap08",
                "irMap09", "irMap10", "irMap11",
                "irMap12", "irMap13", "irMap14",
                "irMap15", "irMap16", "irMap17"
            ],
            [
                "1997", "1998", "1999",
                "2000", "2001", "2002",
                "2003", "2004", "2005",
                "2006", "2007", "2008",
                "2009", "2010", "2011",
                "2012", "2013", "2014",
                "2015", "2016", "2017"
            ]
        )
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Dataset metadata
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    lanid_bnds = [1997, 2017]
    lanid_res = 30

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Get irrigation status for a single year
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    def get_irrig_val(aoi, yod):
        """
        Determine whether irrigation was present within an AOI.
        """

        irrig_year = ee.Number(yod).clamp(
            lanid_bnds[0],
            lanid_bnds[1]
        )

        irrig_img = (
            lanid
            .select(irrig_year.toInt().format())
            .rename("irrig")
        )

        irrig_w_null = ee.Number(
            irrig_img.reduceRegion(
                reducer=ee.Reducer.max(),
                geometry=aoi,
                scale=lanid_res,
                maxPixels=1e13
            ).get("irrig")
        )

        irrig = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(irrig_w_null, None),
                0,
                irrig_w_null
            )
        ).toInt()

        return irrig

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Set AOI and installation year
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    aoi = feature.geometry()
    null_inst_yr = 2022
    inst_yr_raw = feature.get("instYr")

    inst_yr_checked = ee.Algorithms.If(
        ee.Algorithms.IsEqual(inst_yr_raw, None),
        null_inst_yr,
        inst_yr_raw)

    inst_yr = ee.Number(
        ee.Algorithms.If(
            ee.Number(inst_yr_checked).eq(-9999),
            null_inst_yr,
            inst_yr_checked
        ))

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Get irrigation status for multiple years prior
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    irrig_props = {}

    for year_prior in range(1, 6):
        yod = inst_yr.subtract(year_prior)
        irrig_props[f"irrig_yr{year_prior}"] = get_irrig_val(aoi, yod)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Return feature with irrigation attributes
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    return feature.set(irrig_props)

## Run GEE Extraction

In [ ]:
# Call GEE GM-SEUS arrays asset
arraysAsset = ee.FeatureCollection(gmseusGEEAssetPath)

# Set export variables
region = "CONUS"
lulcAttrs = ["lulc_yr1", "lulc_yr2", "lulc_yr3", "lulc_yr4", "lulc_yr5"]
irrigAttrs = ["irrig_yr1", "irrig_yr2", "irrig_yr3", "irrig_yr4", "irrig_yr5"]

# Split for computational efficiency: In order to not exceed the GEE memory limit, we will split the arraysAsset into smaller chunks and process them separately. For now, split by STATEFP
idExportList = (arraysAsset.aggregate_array("STATEFP").distinct().sort().getInfo())
idExportList = [s for s in idExportList if s is not None]
for id in idExportList:
    # Convert from gdf to gee object - only needed if the object is not already a GEE Asset, which it is in this case
    # arraysChunkGDF = arraysAsset[arraysAsset['STATEFP'] == id]
    # arraysChunk = ee.FeatureCollection(arraysAssetGDF.apply(gdf_row_to_ee_feature, axis=1).tolist())

    # Filter the arraysAsset for the current stateFP
    arraysChunk = arraysAsset.filter(ee.Filter.eq("STATEFP", id))

    # Map GEE functions to get LULC and irrigation attributes, then export results to Drive
    out_fc = arraysChunk.map(getGEE_lulc_usa).map(getGEE_irrigStatus_usa)
    task = ee.batch.Export.table.toDrive(
        collection=out_fc,
        description="gmseus_multiLULC_" + region + "_" + id,
        folder='gmseus_multiLULC',
        fileFormat="CSV",
        selectors=[
            "arrayID", "instYr", "version", # base reference attributes
            *lulcAttrs,
            *irrigAttrs
        ]
    )

    task.start()

# Pull Output and get Co-Location Status
*Co-Location*: Agriculture, Natural, Other

Right now, just priorLULC

In [14]:
# Paths
gmseusPath = r"S:\Users\stidjaco\R_files\BigPanel\Data\Derived\GMSEUS\SHP\GMSEUS_Arrays_Final_2025_v2_1.shp"
lulcSchemaPath = r"S:\Users\stidjaco\R_files\BigPanel\Data\Derived\cdlClasses.csv"
uspvdbPath = r"S:\Users\stidjaco\R_files\BigPanel\Data\Downloaded\SolarDB\USPVDB\uspvdb_v4_0_20260414.shp"

# Call gmseus
gmseusArrays = gpd.read_file(gmseusLocalPath)

# Call LUCL Schema (CDL Schema: Value = cdl numeric class 1-255 | 1000 = Cultivated Grassland | 1001 = Natural Grassland | 1002 = Unknown Grassland, Class = cdl class name, gmseusClass = gmseus class name)
lulcSchema = pd.read_csv(lulcSchemaPath)

# Set path 
geeOutPath = f"{gdriveOutputPath}/gmseus_multiLULC"

# Call geeOutPath (folder) all files and concatenate into a single dataframe
geeOutDF = pd.concat([pd.read_csv(os.path.join(geeOutPath, f)) for f in os.listdir(geeOutPath) if f.endswith('.csv')], ignore_index=True)

# Merge totArea from gmseus to geeOutDF using arrayID
geeOutDF = geeOutDF.merge(gmseusArrays[['arrayID', 'totArea']], on='arrayID', how='left')

# Determine co-location status based on occurance of prior land use classes. 
# Preference agricultural co-location, check first. Use schema gmseusClass to determine if the most recent year is "Farmland" or if a majority of the 5 years are ag.
# If the most recent year is "Farmland" from schmea gmseusClass, colocation = "agriculture", else NA.
# If a majority of the 5 years are ag, colocation = "agriculture", else NA.
# Then check for majority gmseusClass (other two options are "Natural" and "Other") and assing status

# From schema, get Value lists for each gmseusClass
farmland_values = lulcSchema[lulcSchema['gmseusClass'] == 'Farmland']['Value'].tolist()
natural_values = lulcSchema[lulcSchema['gmseusClass'] == 'Natural']['Value'].tolist()
other_values = lulcSchema[lulcSchema['gmseusClass'] == 'Other']['Value'].tolist()

# Define a function to assign colocation status based on the most recent year and the majority of the 5 years
def assign_colocation_status(row):
    # Check if the most recent year is farmland (in farmland_values)
    if row['lulc_yr1'] in farmland_values:
        return "agriculture"
    # Check if the majority of the 5 years are farmland
    farmland_count = sum([1 for year in ['lulc_yr1', 'lulc_yr2', 'lulc_yr3', 'lulc_yr4', 'lulc_yr5'] if row[year] in farmland_values])
    if farmland_count >= 3:
        return "agriculture"
    # Check if the majority of the 5 years are natural
    natural_count = sum([1 for year in ['lulc_yr1', 'lulc_yr2', 'lulc_yr3', 'lulc_yr4', 'lulc_yr5'] if row[year] in natural_values])
    if natural_count >= 3:
        return "natural"
    # Check if the majority of the 5 years are other
    other_count = sum([1 for year in ['lulc_yr1', 'lulc_yr2', 'lulc_yr3', 'lulc_yr4', 'lulc_yr5'] if row[year] in other_values])
    if other_count >= 3:
        return "other"
    # Now, check if the most recent year is natural (in natural_values)
    if row['lulc_yr1'] in natural_values:
        return "natural"
    # Now, check if the most recent year is other (in other_values)
    if row['lulc_yr1'] in other_values:
        return "other"
    return "other"

# Apply the function to the DataFrame
geeOutDF['colocation'] = geeOutDF.apply(assign_colocation_status, axis=1)

# Match colocation status to the original gmseusArrays dataframe
gmseusArrays = gmseusArrays.merge(geeOutDF[['arrayID', 'colocation']], on='arrayID', how='left')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Get Other LULC from USPVDB

# Call uspvdb shapefile
uspvdb = gpd.read_file(uspvdbPath)

# Transform uspvdb to toCRS
uspvdb = uspvdb.to_crs(gmseusArrays.crs)

# Set a new column that is uspvdbLULC that is based on p_type
# If p_type is 'superfund', 'AML', 'PCSC', 'landfill', 'RCRA', or 'landfill-named', set LULC column to 'Other LULC'
uspvdb['uspvdbLULC'] = uspvdb.apply(lambda x: 'other' if x['p_type'] in ['superfund', 'AML', 'PCSC', 'landfill', 'RCRA', 'landfill-named'] else x['p_type'], axis=1)

# Drop all rows where uspvdbLULC is not 'Other LULC'
uspvdbOtherLULC = uspvdb[uspvdb['uspvdbLULC'] == 'other'].copy()

# Spatial join between arrays and uspvdbOtherLULC
arraysOtherLULC = gpd.sjoin(gmseusArrays, uspvdbOtherLULC[['uspvdbLULC', 'geometry']], how='left', predicate='intersects')
gmseusArrays = gmseusArrays.merge(arraysOtherLULC[['arrayID', 'uspvdbLULC']], on='arrayID', how='left')

# If uspdbLULC is 'other', set colocation to 'other', else keep colocation as is
gmseusArrays['colocation'] = gmseusArrays.apply(lambda x: 'other' if x['uspvdbLULC'] == 'other' else x['colocation'], axis=1)
gmseusArrays = gmseusArrays.drop(columns=['uspvdbLULC'], errors='ignore')

# Print value counts of colocation status
print(gmseusArrays['colocation'].value_counts())

# Export the updated gmseusArrays with agrisolar and irrigation status to a new shapefile
output_shapefile_path = r"S:\Users\stidjaco\R_files\BigPanel\Data\Derived\intermediateProducts\gmseusArrays_with_colocation.shp"
gmseusArrays.to_file(output_shapefile_path, driver='ESRI Shapefile')

colocation
agriculture    7408
other          6077
natural        5155
Name: count, dtype: int64


# ARCHIVE

In [9]:
# Print number of agrisolar and non-agrisolar arrays
agrisolar_count = geeOutDF['agrisolar'].sum()
non_agrisolar_count = len(geeOutDF) - agrisolar_count   
print(f"Number of agrisolar arrays: {agrisolar_count}")
print(f"Number of non-agrisolar arrays: {non_agrisolar_count}")

# Print number of previously irrigated and non-irrigated arrays
agrisolar = geeOutDF[geeOutDF['agrisolar'] == True]
irrigated_count = agrisolar['irrigated'].sum()
non_irrigated_count = len(agrisolar) - irrigated_count
print(f"Number of previously irrigated arrays: {irrigated_count}")
print(f"Number of non-irrigated arrays: {non_irrigated_count}")

# Print proportion of land agrisolar and proportion of agrisolar land that was previously irrigated (area attribute: totArea)
total_area = geeOutDF['totArea'].sum()
agrisolar_area = agrisolar['totArea'].sum()
proportion_agrisolar = agrisolar_area / total_area if total_area > 0 else 0
print(f"Proportion of agrisolar land: {proportion_agrisolar}")
print(f"Proportion of agrisolar land that was previously irrigated: {irrigated_count / agrisolar_count if agrisolar_count > 0 else 0}")
irrigated_area = agrisolar[agrisolar['irrig_yr1'] == 1]['totArea'].sum()
print(f"Proportion of agrisolar land area that was previously irrigated: {irrigated_area / agrisolar_area if agrisolar_area > 0 else 0}")

# Print total agrisolar area in acres
total_agrisolar_area_acres = agrisolar_area * 0.000247105  # Convert square meters to acres
print(f"Total agrisolar area in acres: {total_agrisolar_area_acres}")

# Print total irrigated agrisolar area in acres
total_irrigated_agrisolar_area_acres = irrigated_area * 0.000247105  # Convert square meters to acres
print(f"Total irrigated agrisolar area in acres: {total_irrigated_agrisolar_area_acres}")

Number of agrisolar arrays: 9210
Number of non-agrisolar arrays: 9338
Number of previously irrigated arrays: 1579
Number of non-irrigated arrays: 7631
Proportion of agrisolar land: 0.5593583883508685
Proportion of agrisolar land that was previously irrigated: 0.1714440825190011
Proportion of agrisolar land area that was previously irrigated: 0.3850809235652411
Total agrisolar area in acres: 510652.15443680505
Total irrigated agrisolar area in acres: 196642.40325110502
